# ============================================================
# STEP 10 — PER-CLASS RESULTS + CONFUSION MATRIX (TUMC)
# ============================================================
# The Results/Discussion sections promise a per-class breakdown
# but do not yet show one. With scam at ~1% prevalence, the
# macro-F1 of 0.897 hides how the rare classes actually perform.
# This script produces, for the deployment configuration
# (inductive graph, Experiment C, HistGB):
#
#   1. Per-class precision / recall / F1 / support (5-class)
#   2. The 5x5 confusion matrix (counts + row-normalised)
#   3. Binary security view: detection rate, false-positive rate
#   4. LaTeX-ready booktabs table for the manuscript
#
# Runs out-of-fold predictions across the 5 domain-insulated
# folds so every URL is predicted by a model that never saw its
# domain — the honest per-class estimate.
#
# Output: step10_perclass.csv, step10_confusion.csv,
#         step10_perclass_table.tex, step10_confusion.png
# ============================================================

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.metrics import matthews_corrcoef

warnings.filterwarnings("ignore")
SEED = 42
DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
for cand in ["features_full_final_inductive_dedup.csv", "features_full_v12.csv"]:
    INPUT = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT): break

LEAKY = ["cluster_malicious_ratio","campaign_token_score","is_tr_domain","is_https"]
CLASS_NAMES = ["benign","phishing","malware","scam","other_malicious"]

print("="*70)
print("STEP 10 — PER-CLASS RESULTS (deployment config: inductive, Exp C)")
print("="*70)

df = pd.read_csv(INPUT, low_memory=False)
META = {"url","source","tld","label","label_enc","class_final",
        "class_enc","fold","reg_domain"}
FEATURES = [c for c in df.columns if c not in META and c not in LEAKY]
y = df["class_enc"].values
folds = df["fold"].values
X = df[FEATURES].fillna(0).values

# Map class_enc → name (verify ordering against your encoder)
enc_to_name = (df[["class_enc","class_final"]].drop_duplicates()
                 .set_index("class_enc")["class_final"].to_dict())
names = [enc_to_name.get(i, f"class{i}") for i in sorted(df["class_enc"].unique())]
print(f"  Classes: {names}")
print(f"  Features: {len(FEATURES)}  Rows: {len(df):,}")

# ── Out-of-fold predictions ──────────────────────────────────
print("\n  Generating out-of-fold predictions (5 domain-insulated folds)...")
oof_pred = np.zeros(len(df), dtype=int)
oof_proba = np.zeros((len(df), len(names)))
for fid in sorted(set(folds)):
    tri = np.where(folds != fid)[0]; tei = np.where(folds == fid)[0]
    m = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05,
        max_iter=300, random_state=SEED)
    m.fit(X[tri], y[tri])
    oof_pred[tei] = m.predict(X[tei])
    oof_proba[tei] = m.predict_proba(X[tei])
    print(f"    fold {fid} done")

# ── Per-class metrics ────────────────────────────────────────
p, r, f1, sup = precision_recall_fscore_support(y, oof_pred, zero_division=0)
per = pd.DataFrame({"class": names, "precision": p, "recall": r,
                    "f1": f1, "support": sup})
per["support_pct"] = per["support"] / per["support"].sum() * 100
print("\n  Per-class metrics:")
print(per.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
macro_f1 = f1.mean()
print(f"\n  Macro-F1: {macro_f1:.4f}")
mcc_5c = matthews_corrcoef(y, oof_pred)
print(f"  MCC (5-class):  {mcc_5c:.4f}")

print(f"  MCC (5-class):  {mcc_5c:.4f}")
per.to_csv(os.path.join(DATA_DIR,"step10_perclass.csv"), index=False)

# ── Confusion matrix ─────────────────────────────────────────
cm = confusion_matrix(y, oof_pred)
cm_df = pd.DataFrame(cm, index=names, columns=names)
cm_df.to_csv(os.path.join(DATA_DIR,"step10_confusion.csv"))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, (a1,a2) = plt.subplots(1,2, figsize=(16,6.5))
for ax, mat, title, fmt in [(a1,cm,"Counts","d"),(a2,cm_norm,"Row-normalised (recall)",".2f")]:
    im = ax.imshow(mat, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(names))); ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_yticklabels(names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontweight="bold")
    thr = mat.max()/2
    for i in range(len(names)):
        for j in range(len(names)):
            v = mat[i,j]
            ax.text(j,i, format(v,fmt), ha="center", va="center",
                    color="white" if v>thr else "black", fontsize=9)
plt.suptitle("TUMC Five-Class Confusion (HistGB, inductive, Exp C)", fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(DATA_DIR,"step10_confusion.png"), dpi=150)
plt.close()
print(f"  Saved confusion matrix → step10_confusion.png")

# ── Binary security view ─────────────────────────────────────
benign_idx = names.index("benign") if "benign" in names else 0
y_bin = (y != benign_idx).astype(int)
pred_bin = (oof_pred != benign_idx).astype(int)
tn = ((y_bin==0)&(pred_bin==0)).sum(); fp = ((y_bin==0)&(pred_bin==1)).sum()
fn = ((y_bin==1)&(pred_bin==0)).sum(); tp = ((y_bin==1)&(pred_bin==1)).sum()
det_rate = tp/(tp+fn); fpr = fp/(fp+tn)
print(f"\n  Binary security view (malicious detection):")
print(f"    Detection rate (recall): {det_rate:.4f}")
print(f"    False-positive rate:     {fpr:.4f}")
print(f"    Missed attacks (FN):     {fn:,}")
print(f"    False alarms (FP):       {fp:,}")

# ── LaTeX table ──────────────────────────────────────────────
tex = ["\\begin{table}[htbp]","\\centering",
 "\\caption{Per-class performance of the reference classifier (HistGB, inductive graph, Experiment~C), computed from out-of-fold predictions across the five domain-insulated folds. Support is the number of URLs in each class.}",
 "\\label{tab:perclass}","\\footnotesize","\\begin{tabular}{lcccr}","\\toprule",
 "Class & Precision & Recall & $F_1$ & Support \\\\","\\midrule"]
for _, row in per.iterrows():
    tex.append(f"{row['class'].replace('_','-')} & {row['precision']:.3f} & "
               f"{row['recall']:.3f} & {row['f1']:.3f} & {int(row['support']):,} \\\\")
tex += ["\\midrule",
        f"Macro avg & {p.mean():.3f} & {r.mean():.3f} & {f1.mean():.3f} & {int(sup.sum()):,} \\\\",
"\\midrule",
f"\\multicolumn{{4}}{{l}}{{Matthews Correlation Coefficient (MCC)}} & {mcc_5c:.3f} \\\\",
"\\bottomrule",
 "\\end{tabular}","\\end{table}"]
with open(os.path.join(DATA_DIR,"step10_perclass_table.tex"),"w") as fh:
    fh.write("\n".join(tex))
print(f"  Saved LaTeX table → step10_perclass_table.tex")

print(f"\n{'='*70}\nSTEP 10 COMPLETE\n{'='*70}")
print("  Honest reviewer-facing reading: report per-class recall so the")
print("  rare-class (scam) performance is explicit, not hidden in macro-F1.")

STEP 10 — PER-CLASS RESULTS (deployment config: inductive, Exp C)
  Classes: ['benign', 'phishing', 'malware', 'scam', 'other_malicious']
  Features: 135  Rows: 1,239,308

  Generating out-of-fold predictions (5 domain-insulated folds)...
    fold 0 done
    fold 1 done
    fold 2 done
    fold 3 done
    fold 4 done

  Per-class metrics:
          class  precision  recall     f1  support  support_pct
         benign     0.9620  0.9579 0.9599   190740      15.3908
       phishing     0.9412  0.8887 0.9142   512802      41.3781
        malware     0.9723  0.8318 0.8966    96768       7.8082
           scam     0.8914  0.8323 0.8608    13480       1.0877
other_malicious     0.8504  0.9389 0.8925   425518      34.3351

  Macro-F1: 0.9048
  MCC (5-class):  0.8710
  MCC (5-class):  0.8710
  Saved confusion matrix → step10_confusion.png

  Binary security view (malicious detection):
    Detection rate (recall): 0.9931
    False-positive rate:     0.0421
    Missed attacks (FN):     7,222
   